# Notebook 03: Explainability, Grad-CAM & Systematic Error Analysis
**Project:** COVID-CT Deep Learning Classifier (`covid-ct-dashboard`)


> [!WARNING]
> **EDUCATIONAL & PORTFOLIO PROJECT ONLY • NOT A MEDICAL DEVICE**  
> *This notebook, associated models, and data pipelines are strictly for research, algorithmic benchmarking, and portfolio demonstration. They are NOT certified by the FDA, CE, or any medical regulatory agency and must NEVER be used for clinical decision making, diagnosis, triage, or patient treatment.*


### Objectives:
1. Extract top-20 most confident prediction errors.
2. Analyze misclassification transitions (e.g. `pCT -> nCT`, `nCT -> pCT`).
3. Generate Grad-CAM heatmaps overlaying salient feature activations.
4. **Discussion of Likely Failure Causes**: Motion artifact, peripheral ground-glass similarity, and partial-volume slice boundaries.
5. **Honest Evaluation Note**: Methodological impact of proxy group splitting vs. volumetric 3D CT.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

PROJECT_ROOT = Path('.').resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import POWERBI_DATA_DIR, FIGURES_DIR, CLASS_NAMES
pred_csv = POWERBI_DATA_DIR / 'predictions.csv'
df_preds = pd.read_csv(pred_csv)
print('Loaded predictions:', len(df_preds))

## 1. Top-20 Most Confident Errors

In [ ]:
test_errors = df_preds[(df_preds['split'] == 'test') & (df_preds['correct'] == 0)].copy()
top20 = test_errors.sort_values(by='conf', ascending=False).head(20)
display(top20[['model', 'image_id', 'true_label', 'pred_label', 'conf', 'error_type', 'mean_brightness']])

## 2. Misclassification Pair Breakdown

In [ ]:
err_pairs = test_errors.groupby(['model', 'error_type']).size().unstack(fill_value=0)
display(err_pairs)

test_errors['error_type'].value_counts().plot(kind='barh', color='#F87171', figsize=(7, 4))
plt.title('Misclassification Transitions on Test Set', fontweight='bold')
plt.xlabel('Count')
plt.grid(True, linestyle='--', alpha=0.3)
plt.show()

## 3. Clinical & Methodological Discussion

### Likely Causes of Systematic Errors:
1. **Peripheral Ground-Glass Opacities vs. Atelectasis**: Subpleural dependent density in normal elderly patients frequently mimics early multifocal COVID-19 consolidations.
2. **Motion and Diaphragmatic Artifacts**: Slices acquired during shallow respiration display cardiac motion blur that distorts lung parenchyma textures.
3. **Non-informative (NiCT) Slice Boundaries**: Transitional slices between the apex and upper thoracic soft tissues have lower lung area, creating borderline ambiguity between NiCT and normal nCT.

### Honest Appraisal of Split Strategy & Score Inflation:
- In standard benchmarks where random slice splitting is used, test accuracies often exceed 98-99% because near-duplicate adjacent slices from the same scan volume exist in both training and test sets.
- Enforcing proxy grouping reduces this inflation, providing a more realistic assessment of out-of-scan generalization.
- However, since the dataset did not provide native DICOM patient headers, sequence proxy clustering is an approximation. True prospective clinical performance on heterogeneous scanners across different hospitals would be expected to experience further distribution shift.